In [1]:
from google.colab import drive
drive.mount('/content/drive',force_remount=1)

Mounted at /content/drive


In [2]:
!pip install -U -q bitsandbytes accelerate transformers peft trl unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from unsloth import FastLanguageModel
import torch
import json
from datasets import Dataset

max_seq_length = 16384 # Tăng lên để chứa các file code dài
dtype = None # Tự động phát hiện (T4 dùng Float16)
load_in_4bit = True # Tiết kiệm VRAM

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Thêm LoRA Adapters (Unsloth tối ưu hóa các lớp này để train nhanh gấp 2x)
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 128,
    lora_dropout = 0, # Tối ưu nhất cho Unsloth
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Cực kỳ quan trọng để tránh OOM
    random_state = 3407,
)

In [ ]:
all_data = []
with open("/content/drive/MyDrive/PBL5/final_dataset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        all_data.append(item)
        
        # Lọc mẫu mẫu positive (những mẫu KHÔNG mang nhãn 'none')
        try:
            out_js = json.loads(item.get("output", "{}"))
            is_none_class = out_js.get("final_decision", {}).get("label") == "none"
        except:
            is_none_class = False
        
        if not is_none_class:
            # Oversampling x10
            for _ in range(9):
                all_data.append(item)

dataset = Dataset.from_list(all_data)
print(dataset[0])

def formatting_prompts_func(examples):
    instructions = examples.get("instruction", [])
    inputs = examples.get("input", [])
    outputs = examples.get("output", [])
    texts = []

    for instr, user_input, out in zip(instructions, inputs, outputs):
        messages = [
            {"role": "system", "content": instr},
            {"role": "user", "content": user_input},
            {"role": "assistant", "content": out}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)

    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched=True)


In [ ]:
print(f"Số lượng mẫu trong dataset: {len(dataset)}")
print(f"Các cột hiện có: {dataset.column_names}")

if len(dataset) > 0:
    # Kiểm tra thử nội dung mẫu đầu tiên
    first_sample = dataset[0]
    print("Nội dung mẫu đầu tiên:")
    # Thay 'text' bằng tên cột bạn dùng trong dataset_text_field
    print(first_sample.get("text", "CỘT 'text' KHÔNG TỒN TẠI!"))
else:
    print("CẢNH BÁO: Dataset của bạn đang rỗng!")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
def clean_diff(diff_text):
    # Chỉ giữ lại các dòng tiêu đề diff và các dòng có dấu + hoặc -
    lines = diff_text.split('\n')
    cleaned = [l for l in lines if l.startswith(('+', '-', '@@'))]
    return '\n'.join(cleaned)

# Chế độ suy luận nhanh
FastLanguageModel.for_inference(model)

prompt = dataset[0]["text"].split("<|im_start|>assistant")[0] + "<|im_start|>assistant"

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 2048)